# Claude Agent SDK: input modes (streaming input vs single message)

There are **two ways to send a prompt** to an agent, and they behave very differently:

| | **Single message input** | **Streaming input** (recommended) |
|---|---|---|
| You pass | a plain **string** | an **async generator** of messages (or use `ClaudeSDKClient`) |
| Shape | one-shot: ask, get an answer | a **long-lived, interactive** session |
| Images | no | **yes** |
| Queue more messages | no | **yes** (they process in order) |
| Interrupt mid-task | no | **yes** |
| Multi-turn feel | via `continue_conversation` / `resume` | natural, one live session |
| Best for | stateless jobs, lambdas, CI, simple scripts | chat UIs, apps with follow-ups, approvals, attachments |

**What you will learn (and see running):**
1. Single message input and its limits
2. Streaming input: an async generator of user messages
3. Sending an **image** to the agent
4. Using `ClaudeSDKClient` for interactive turns
5. **Interrupting** a running agent
6. The silent-failure trap in generators, and how to defend against it

## The mental model

```
 SINGLE MESSAGE                         STREAMING INPUT
 ──────────────                         ───────────────
 query("do X") ──▶ agent ──▶ result     your app ──msg 1──▶ ┐
                                        your app ──msg 2──▶ │ agent (stays alive)
 (new call = new run; use                your app ──image──▶ │   ◀── responses stream back
  continue/resume for context)           your app ──interrupt▶ ┘
```

Streaming input lets the agent run as a **long-lived process** that takes user input, handles interruptions, surfaces permission requests and manages the session for you.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("input-modes-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Demo project

A tiny app with an authentication function and an authorization function, so we have something to ask questions about.

In [ ]:
APP_PY = """def login(username, password, users):
    \"\"\"Authentication: prove WHO the user is.\"\"\"
    return users.get(username) == password


def can_delete(user_role):
    \"\"\"Authorization: decide WHAT the user may do.\"\"\"
    return user_role in {"admin", "owner"}
"""

(DEMO_DIR / "app.py").write_text(APP_PY)
print("Created app.py")

In [ ]:
import asyncio
from claude_agent_sdk import (
    query, ClaudeAgentOptions, ClaudeSDKClient,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

def short(value, n=140):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"

## 2. Single message input

You pass a **string**. The agent runs to completion and the stream ends. This is perfect when you need a one-shot answer or run in a stateless environment such as a lambda function.

**Limits** (single message mode does *not* support):
- direct image attachments
- dynamic message queueing
- real-time interruption
- natural multi-turn conversations

Two details to remember from earlier notebooks: a single-shot `query()` **raises** after yielding an error result (like `error_max_turns`), so wrap it in `try`; and to talk *again* you must **continue or resume** the session explicitly.

In [ ]:
async def single_message_example():
    # One-shot query: the prompt is a plain string.
    try:
        async for message in query(
            prompt="Explain the authentication flow in app.py in one sentence.",
            options=ClaudeAgentOptions(max_turns=5, allowed_tools=["Read", "Grep"], cwd=str(DEMO_DIR)),
        ):
            if isinstance(message, ResultMessage) and message.subtype == "success":
                print("1st answer:", short(message.result))
    except Exception as e:
        print(f"Query failed: {e}")

    # A second query is a NEW run. continue_conversation=True reattaches the last session.
    try:
        async for message in query(
            prompt="Now explain the authorization process in one sentence.",
            options=ClaudeAgentOptions(continue_conversation=True, max_turns=5, cwd=str(DEMO_DIR),
                                       allowed_tools=["Read", "Grep"]),
        ):
            if isinstance(message, ResultMessage) and message.subtype == "success":
                print("2nd answer:", short(message.result))
    except Exception as e:
        print(f"Query failed: {e}")

await single_message_example()

## 3. Streaming input: an async generator of messages

Instead of a string you pass an **async generator** that *yields user messages over time*. Each message is a dict:

```python
{"type": "user", "message": {"role": "user", "content": "your text here"}}
```

`content` is either a **string** or a **list of content blocks** (text, image...). Because it is a generator, your code can `await` anything between messages: user typing, a webhook, a timer.

Let's build the message helpers and test them with **no API calls**.

In [ ]:
# OFFLINE
import base64

def user_text(text):
    """A plain-text user message for streaming input."""
    return {"type": "user", "message": {"role": "user", "content": text}}


def user_text_and_image(text, image_path, media_type="image/png"):
    """A user message that carries text AND a base64-encoded image."""
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    return {
        "type": "user",
        "message": {
            "role": "user",
            "content": [
                {"type": "text", "text": text},
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": data}},
            ],
        },
    }


print(user_text("hello"))

### Make an image to send
To send an image we need a file. The next cell writes a small **PNG using only the standard library**: three coloured boxes (red, green, blue) joined by black lines, like a trivial architecture diagram.

In [ ]:
# OFFLINE
import zlib, struct

def write_png(path, width, height, pixel):
    """Write an RGB PNG. `pixel(x, y)` returns an (r, g, b) tuple."""
    rows = b"".join(
        b"\x00" + b"".join(bytes(pixel(x, y)) for x in range(width)) for y in range(height)
    )
    def chunk(tag, data):
        body = tag + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body) & 0xFFFFFFFF)
    png = (b"\x89PNG\r\n\x1a\n"
           + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
           + chunk(b"IDAT", zlib.compress(rows))
           + chunk(b"IEND", b""))
    Path(path).write_bytes(png)


def diagram_pixel(x, y):
    boxes = [((20, 80), (220, 50, 50)), ((120, 180), (50, 180, 70)), ((220, 280), (50, 90, 220))]
    if 30 <= y < 90:
        for (x0, x1), colour in boxes:
            if x0 <= x < x1:
                return colour
        if 58 <= y <= 61 and 80 <= x < 220:
            return (0, 0, 0)          # connecting lines
    return (255, 255, 255)


write_png(DEMO_DIR / "diagram.png", 300, 120, diagram_pixel)
print("diagram.png:", (DEMO_DIR / "diagram.png").stat().st_size, "bytes")
msg = user_text_and_image("Describe this image.", DEMO_DIR / "diagram.png")
print("content blocks:", [b["type"] for b in msg["message"]["content"]])

### Streaming text, with a pause between messages
`query()` also accepts an async generator as `prompt`. We yield one message, simulate the user thinking for a second, then yield a follow-up. Both are processed **in order, inside one live session**.

In [ ]:
async def two_messages():
    yield user_text("Read app.py and list the function names. One line, no explanation.")
    await asyncio.sleep(1)        # your app could be waiting for the user here
    yield user_text("Which of those functions decides what a user may do? One sentence.")


results = 0
async for message in query(
    prompt=two_messages(),
    options=ClaudeAgentOptions(allowed_tools=["Read"], cwd=str(DEMO_DIR), max_turns=6),
):
    if isinstance(message, ResultMessage):
        results += 1
        print(f"result #{results} [{message.subtype}]:", short(message.result))
print("result messages seen:", results)

## 4. Interactive turns with `ClaudeSDKClient` (and an image)

`ClaudeSDKClient` keeps **one session open**. Each `await client.query(...)` sends a message (a string *or* a generator), and `client.receive_response()` yields messages **until the first result**.

> **Important detail.** `receive_response()` ends at the first `ResultMessage`. To read several responses, use **one `query()` + `receive_response()` pair per message**, not one pair for a generator that yields several messages.

In [ ]:
def show(message):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print("  Claude:", short(block.text, 300))
            elif isinstance(block, ToolUseBlock):
                print("  [tool]", block.name)
    elif isinstance(message, ResultMessage):
        print(f"  [done: {message.subtype}]")


async def one_message(content_message):
    yield content_message


options = ClaudeAgentOptions(allowed_tools=["Read"], cwd=str(DEMO_DIR), max_turns=6)

async with ClaudeSDKClient(options=options) as client:
    print("Turn 1: text")
    await client.query("Read app.py. How many functions does it define? Answer with just a number.")
    async for m in client.receive_response():
        show(m)

    print("Turn 2: an image, in the SAME session")
    await client.query(one_message(user_text_and_image(
        "What colours are the three boxes, left to right, and what joins them?",
        DEMO_DIR / "diagram.png",
    )))
    async for m in client.receive_response():
        show(m)

The second turn only works because the session is **streaming**: a single-message `query("...")` string cannot carry an image.

> If an image block's `source` is missing or not an object, the SDK does **not** raise. Claude Code sends Claude a text note such as `[Image could not be processed: ...]` and carries on.

## 5. Interrupting the agent

With a live client you can send a **cancel signal** while the agent works. That is what makes a "Stop" button in a chat UI possible.

We ask for a very long answer, start reading the response, and call `client.interrupt()` after a few seconds. (Exact timing varies, so treat this as a demonstration. The result subtype after an interrupt is often `error_during_execution` rather than `success`, and that is normal: the run was cut short on purpose. What matters is that the session stays usable afterwards.)

In [ ]:
async with ClaudeSDKClient(options=ClaudeAgentOptions(cwd=str(DEMO_DIR), max_turns=3)) as client:
    await client.query("Write a very long, detailed essay (at least 1500 words) on the history of compilers.")

    async def stop_soon():
        await asyncio.sleep(4)
        print("\n  >>> interrupt() called")
        await client.interrupt()

    stopper = asyncio.create_task(stop_soon())
    chars = 0
    async for m in client.receive_response():
        if isinstance(m, AssistantMessage):
            for block in m.content:
                if isinstance(block, TextBlock):
                    chars += len(block.text)
        elif isinstance(m, ResultMessage):
            print(f"  [result: {m.subtype}] text received so far: {chars} chars")
    await stopper

    # The session is still alive after an interrupt.
    await client.query("In one short sentence: what were you doing just now?")
    async for m in client.receive_response():
        show(m)

## 6. The silent-failure trap

If **your generator raises**, you may not get a clear error:

- **Python:** the exception is logged at *debug* level and the session **stalls without raising**. A hung stream with no output usually means a broken generator.
- **TypeScript:** the stream ends with the misleading error `Claude Code process aborted by user`, sometimes after a long minified source line, so read to the end.

Defense: wrap your generator so every failure is **printed loudly** and re-raised. Test it offline:

In [ ]:
# OFFLINE
import traceback

async def loud(gen):
    """Wrap a message generator so its exceptions are never swallowed silently."""
    try:
        async for item in gen:
            yield item
    except Exception:
        print("!! message generator failed:")
        traceback.print_exc()
        raise


async def broken():
    yield user_text("first message is fine")
    open("this-file-does-not-exist.png", "rb")      # the bug
    yield user_text("never reached")


seen = []
try:
    async for m in loud(broken()):
        seen.append(m["message"]["content"])
except FileNotFoundError:
    print("caught it:", seen)

Use it like `query(prompt=loud(my_generator()), ...)`. You also have two other options for debugging: enable Python debug logging (`logging.basicConfig(level=logging.DEBUG)`), or check the code inside the generator first.

## 7. Choosing a mode

| You need... | Use |
|---|---|
| A one-shot answer, a cron job, a lambda | Single message (string) |
| A follow-up later, in a stateless setting | Single message + `continue_conversation` / `resume` |
| Image attachments | **Streaming input** |
| Several messages queued behind each other | **Streaming input** |
| A Stop button / mid-task redirection | **Streaming input** + `interrupt()` |
| Approval prompts and user questions (`can_use_tool`) | **Streaming input** (see the next notebook) |
| A chat UI with live state | `ClaudeSDKClient` |

## 8. Recap

- Single message = **string in, answer out**; simple and stateless, but no images, queueing, or interrupts.
- Streaming input = an **async generator** of `{"type": "user", "message": {...}}` dicts, or a `ClaudeSDKClient`.
- Message `content` can be a string or a **list of blocks** (text + image).
- `receive_response()` stops at the **first result**: one pair per message.
- `client.interrupt()` cancels work; the session survives.
- Generator bugs fail **quietly**: wrap them.

### Try it yourself
1. Send a second image of a different colour layout and ask the agent to compare the two.
2. Queue three messages in `two_messages()` and see how many results come back.
3. Replace the `asyncio.sleep` with `input()` to make a tiny terminal chat.
4. Next: **`agent-sdk-approvals-and-user-input-explained.ipynb`**.